## **rm_instance.py**

Everything so far has been frame-by-frame. Detection looks at one frame.
Association matches boxes within one frame. Neither remembers anything But a violation isn't a property of a frame it's a property of a *rider*,
accumulated over the time they were visible. Something has to hold that
accumulating record, and that's what this file is.

It's the pipeline's memory. Two small classes: one record per rider, and a manager that keeps track of all of them.

## **Setup**

In [1]:
!git clone -q https://github.com/shreyamali17/helmet-violation-detection.git
%cd helmet-violation-detection/project/pipeline
!pip install -q ultralytics scipy

/content/helmet-violation-detection/project/pipeline
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 7.2 MB/s eta 0:00:00


## **Record**

`RMInstance` is one rider–motorcycle pairing and everything the pipeline will
eventually learn about it.

Most of its fields start as `None` and get filled in later by other stages : `temporal.py` writes the helmet status, `association.py` supplies the plate.
The object is created early and completed gradually.

In [2]:
from collections import Counter, deque


class RMInstance:
    def __init__(self, motorcycle_id, rider_id, buffer_size=5):
        self.rider_id = rider_id
        self.motorcycle_id = motorcycle_id

        # rolling window of recent motorcycle assignments
        self._moto_votes = deque(maxlen=buffer_size)
        self._moto_votes.append(motorcycle_id)

        # filled in later by other stages
        self.helmet_status = None
        self.confidence = None
        self.plate_number = None
        self.plate_box = None
        self.is_violation = None

    def vote_motorcycle(self, motorcycle_id):
        self._moto_votes.append(motorcycle_id)
        self.motorcycle_id = Counter(self._moto_votes).most_common(1)[0][0]

    def __repr__(self):
        return (f"RMInstance(moto={self.motorcycle_id}, rider={self.rider_id}, "
                f"helmet={self.helmet_status}, plate={self.plate_number}, "
                f"violation={self.is_violation})")

### **Voting Buffer**

This is the interesting part of the class. Association runs fresh on every frame, and it doesn't always return the same answer. In dense traffic, a rider's box might overlap two motorcycles, and which one wins can flip between frames as boxes shift by a few pixels.

Taking the latest answer would make `motorcycle_id` unstable : flickering
between bikes as the video plays. Instead, the last five assignments are kept and the majority wins.

`deque(maxlen=5)` does the windowing automatically: push a sixth item and the oldest falls off. No manual trimming.

In [3]:
inst = RMInstance(motorcycle_id=12, rider_id=7)
print(f"start:        moto={inst.motorcycle_id}   votes={list(inst._moto_votes)}")

# association wobbles for a couple of frames, then settles
for m in [12, 15, 12, 12, 15, 12, 12]:
    inst.vote_motorcycle(m)
    print(f"saw {m} ->     moto={inst.motorcycle_id}   votes={list(inst._moto_votes)}")

start:        moto=12   votes=[12]
saw 12 ->     moto=12   votes=[12, 12]
saw 15 ->     moto=12   votes=[12, 12, 15]
saw 12 ->     moto=12   votes=[12, 12, 15, 12]
saw 12 ->     moto=12   votes=[12, 12, 15, 12, 12]
saw 15 ->     moto=12   votes=[12, 15, 12, 12, 15]
saw 12 ->     moto=12   votes=[15, 12, 12, 15, 12]
saw 12 ->     moto=12   votes=[12, 12, 15, 12, 12]


Notice that a single frame reporting motorcycle 15 never changes the answer.
It takes a sustained majority to move it and once the window has rolled
past, old votes stop counting at all.

That last property matters. A rider genuinely *can* change motorcycles between
appearances, and a buffer that remembered forever would take a long time to catch up. Five frames is short enough to adapt, long enough to ignore noise.

Watch what a real switch looks like.

In [4]:
inst = RMInstance(motorcycle_id=12, rider_id=7)
for m in [12, 12, 12]:
    inst.vote_motorcycle(m)
print(f"settled on {inst.motorcycle_id}")

print("\nnow the rider is consistently seen on motorcycle 20:")
for m in [20, 20, 20, 20, 20]:
    inst.vote_motorcycle(m)
    print(f"  votes={list(inst._moto_votes)} -> moto={inst.motorcycle_id}")

settled on 12

now the rider is consistently seen on motorcycle 20:
  votes=[12, 12, 12, 12, 20] -> moto=12
  votes=[12, 12, 12, 20, 20] -> moto=12
  votes=[12, 12, 20, 20, 20] -> moto=20
  votes=[12, 20, 20, 20, 20] -> moto=20
  votes=[20, 20, 20, 20, 20] -> moto=20


## **Manager**

One `RMInstance` per rider. The manager's job is to hand you the right one.

`get_or_create` is called every single frame for every matched rider, and
carries the whole design in two branches: if this rider is new, start a record;
if not, add a vote to the existing one.

In [5]:
class RMInstanceManager:
    def __init__(self):
        self.instances_by_rider = {}

    def get_or_create(self, rider_id, motorcycle_id):
        if rider_id not in self.instances_by_rider:
            self.instances_by_rider[rider_id] = RMInstance(
                motorcycle_id=motorcycle_id, rider_id=rider_id)
        else:
            self.instances_by_rider[rider_id].vote_motorcycle(motorcycle_id)
        return self.instances_by_rider[rider_id]

    def all_instances(self):
        return list(self.instances_by_rider.values())

    def summary(self):
        total = len(self.instances_by_rider)
        with_helmet_status = sum(1 for i in self.instances_by_rider.values() if i.helmet_status)
        violations = sum(1 for i in self.instances_by_rider.values() if i.is_violation)
        return {"total_instances": total,
                "confident_helmet_status": with_helmet_status,
                "confirmed_violations": violations}

## **Keyed by rider, not by pair**

The dictionary is `instances_by_rider`, not `instances_by_pair`. One record per
rider, and the motorcycle is a mutable field inside it.

That follows from what the system is actually deciding. The violation belongs
to the rider they're the one not wearing a helmet. Which bike they're on is a
detail attached to them, useful mainly for finding the license plate.

Key by pair and a rider who switches bikes becomes two separate records, each
with half the evidence. Neither might reach `MIN_OBSERVATIONS`, and a real
violation would slip through.

### **`get_or_create` and the tracker**

This whole design leans on one assumption: **track IDs are stable**.

ByteTrack gives rider 7 the ID 7 in every frame they appear. That's what makes
`rider_id` usable as a dictionary key across the whole video.

When the tracker fails a rider is occluded by a truck for a second and comes
back as ID 431 : the pipeline sees a brand new rider with no history. The old
record stays behind, frozen with whatever evidence it had.

That's the main reason `total_instances` runs higher than the number of actual
people in the video.

## **On real video**

Run the pipeline's first two stages and watch the manager fill up.

In [6]:
from ultralytics import YOLO
import config
from detection import get_frame_detections
from association import associate_riders_to_motorcycles

model = YOLO(config.MODEL_PATH)

results = model.track(
    source=config.VIDEO_PATH,
    classes=list(config.CLASS_NAMES.keys()),
    tracker=config.TRACKER,
    persist=True, stream=True, verbose=False,
)

manager = RMInstanceManager()
FRAME_LIMIT = 300

for i, r in enumerate(results):
    if i >= FRAME_LIMIT:
        break
    det, _ = get_frame_detections(r)
    for rider_id, moto_id in associate_riders_to_motorcycles(det["rider"], det["motorcycle"]):
        manager.get_or_create(rider_id, moto_id)

    if i in (0, 50, 100, 200, 299):
        print(f"frame {i:>4}: {len(manager.instances_by_rider)} riders tracked")

print()
print(manager.summary())

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 2 packages in 196ms
Prepared 1 package in 26ms
Installed 1 package in 1ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.6s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

frame    0: 6 riders tracked
frame   50: 16 riders tracked
frame  100: 21 riders tracked
frame  200: 27 riders tracked
frame  299: 39 riders tracked

{'total_instances': 39, 'confident_helmet_status': 0, 'confirmed_violations': 0}


`confident_helmet_status` and `confirmed_violations` are both zero, which is
correct — nothing has run the helmet logic yet. The manager is holding the
records; `temporal.py` is what fills those fields in.

A look at some individual records.

In [7]:
for inst in manager.all_instances()[:8]:
    print(inst)

RMInstance(moto=3, rider=1, helmet=None, plate=None, violation=None)
RMInstance(moto=172, rider=4, helmet=None, plate=None, violation=None)
RMInstance(moto=33, rider=6, helmet=None, plate=None, violation=None)
RMInstance(moto=5, rider=7, helmet=None, plate=None, violation=None)
RMInstance(moto=17, rider=11, helmet=None, plate=None, violation=None)
RMInstance(moto=15, rider=14, helmet=None, plate=None, violation=None)
RMInstance(moto=67, rider=32, helmet=None, plate=None, violation=None)
RMInstance(moto=81, rider=79, helmet=None, plate=None, violation=None)


Every field except the IDs is still `None`. This is the skeleton that the
remaining stages write into.

## **How many are real**

`total_instances` counts tracker IDs, not people. Riders at the edge of the
frame, riders who appear for four frames, riders who get a new ID after an
occlusion all count.

Grouping by how long each was tracked shows the distribution.

In [8]:
# re-run, this time counting appearances per rider
results = model.track(
    source=config.VIDEO_PATH,
    classes=list(config.CLASS_NAMES.keys()),
    tracker=config.TRACKER,
    persist=True, stream=True, verbose=False,
)

appearances = {}
for i, r in enumerate(results):
    if i >= FRAME_LIMIT:
        break
    det, _ = get_frame_detections(r)
    for rider_id in det["rider"]:
        appearances[rider_id] = appearances.get(rider_id, 0) + 1

buckets = {"1-2 frames": 0, "3-9 frames": 0, "10-29 frames": 0, "30+ frames": 0}
for n in appearances.values():
    if n <= 2:    buckets["1-2 frames"] += 1
    elif n < 10:  buckets["3-9 frames"] += 1
    elif n < 30:  buckets["10-29 frames"] += 1
    else:         buckets["30+ frames"] += 1

print(f"{len(appearances)} distinct rider IDs seen\n")
for k, v in buckets.items():
    print(f"  {k:<14} {v:>3}  {'#' * v}")

67 distinct rider IDs seen

  1-2 frames       9  #########
  3-9 frames      17  #################
  10-29 frames    17  #################
  30+ frames      24  ########################


The one- and two-frame IDs are mostly noise: a partial rider at the edge of
the frame, or a detection that didn't hold.

This is exactly what `MIN_OBSERVATIONS = 3` filters out in the next stage. The
manager deliberately keeps everything — throwing data away early is hard to
undo and lets the decision logic apply the standard.

## **Try it**

**Change the buffer size.** `RMInstance(..., buffer_size=1)` means the latest
frame always wins. How much does `motorcycle_id` flicker? Try 15 : does it get
too slow to follow a real switch?

**Break the tie.** Feed `vote_motorcycle` an even split : `[12, 20, 12, 20]`
and see which one `Counter.most_common` picks. Is that behaviour you'd want to
rely on?

**Count the short-lived IDs.** In the histogram above, what fraction of tracked
riders appear for fewer than three frames? That's the fraction the next stage
will refuse to judge.

## **Next**

The manager holds a record for every rider. What it doesn't do is decide
anything : `helmet_status`, `confidence` and `is_violation` are all still
`None`.

`temporal.py` is what looks at a rider's accumulated observations and works out
whether there's enough evidence to make a call.